In [9]:
import torch
import os
import numpy as np
from pathlib import Path
from tqdm import tqdm

from pymatgen.core.structure import Structure
from pymatgen.core.lattice import Lattice
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.io.cif import CifWriter

def main(pt_file, symprec=0.1):
    # 解析并检查文件路径
    pt_path = Path(pt_file).resolve()
    if not pt_path.exists():
        raise FileNotFoundError(f"找不到文件: {pt_path}")

    print(f"正在从 {pt_path} 加载数据...")
    # 加载模型生成结果，设置 weights_only=False 允许加载 Namespace 等对象
    data = torch.load(pt_path, map_location='cpu', weights_only=False)

    frac_coords = data['frac_coords']
    num_atoms = data['num_atoms']
    atom_types = data['atom_types']
    lengths = data['lengths']
    angles = data['angles']

    # 按照每个晶体的原子数量，将拼接的 Tensor 切分为列表
    num_atoms_list = num_atoms.tolist()
    frac_coords_list = torch.split(frac_coords, num_atoms_list)
    atom_types_list = torch.split(atom_types, num_atoms_list)

    structures_with_sym = []

    print("正在重建晶体结构并计算对称性(空间群)...")
    for i in tqdm(range(len(num_atoms_list))):
        a, b, c = lengths[i].tolist()
        alpha, beta, gamma = angles[i].tolist()
        try:
            lattice = Lattice.from_parameters(a, b, c, alpha, beta, gamma)
            species = atom_types_list[i].tolist()
            coords = frac_coords_list[i].tolist()
            struct = Structure(lattice, species, coords)

            # 分析对称性
            sga = SpacegroupAnalyzer(struct, symprec=symprec)
            sg_num = sga.get_space_group_number()
            sg_symbol = sga.get_space_group_symbol()

            structures_with_sym.append({
                'index': i,
                'structure': struct,
                'sg_num': sg_num,
                'sg_symbol': sg_symbol
            })
        except Exception:
            pass

    # 按照空间群编号从低到高排序 (从小到大)
    structures_with_sym.sort(key=lambda x: x['sg_num'])

    total_count = len(structures_with_sym)
    if total_count < 4:
        print(f"警告：有效结构总数仅为 {total_count}，将全部提取。")
        selected_indices = list(range(total_count))
    else:
        # 选取四个索引：最低(0), 1/3处, 2/3处, 最高(最后)
        selected_indices = [
            0,
            total_count // 3,
            (2 * total_count) // 3,
            total_count - 1
        ]

    # 创建目标文件夹
    out_dir = pt_path.parent / "uncondiction_example"
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n已按对称性从低到高排序，正在提取 4 个代表性结构至: {out_dir}\n")

    labels = ["LowestSym", "LowMidSym", "HighMidSym", "HighestSym"]

    for i, idx_in_sorted in enumerate(selected_indices):
        item = structures_with_sym[idx_in_sorted]
        orig_idx = item['index']
        sg_num = item['sg_num']
        sg_symbol = item['sg_symbol']
        struct = item['structure']
        label = labels[i]

        # 格式化文件名
        safe_symbol = str(sg_symbol).replace('/', '_')
        filename = f"{label}_sg{sg_num}_{safe_symbol}_idx{orig_idx}.cif"
        out_filepath = out_dir / filename

        # 写入文件
        CifWriter(struct).write_file(str(out_filepath))
        print(f"[{label}] 空间群: {sg_num} ({sg_symbol}) -> {filename}")


if __name__ == '__main__':
    # ==========================================
    # 请在这里直接修改你的参数
    # ==========================================

    # 你的 pt 文件路径
    PT_FILE_PATH = "../output/singlerun/2026-03-10/15-50-41-mp_20/eval_gen_mp_20.pt"

    # 计算对称性的容差
    SYMPREC = 0.1

    # ==========================================

    main(pt_file=PT_FILE_PATH, symprec=SYMPREC)

正在从 E:\WORKSPACE\CodePlace\test_CGDiT\output\singlerun\2026-03-10\15-50-41-mp_20\eval_gen_mp_20.pt 加载数据...
正在重建晶体结构并计算对称性(空间群)...


100%|██████████| 10000/10000 [00:21<00:00, 469.75it/s]
D:\app\anaconda3\envs\cgdit\lib\site-packages\pymatgen\core\composition.py:1366: UserWarning: No Pauling electronegativity for Ne. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  syms: list[str] = sorted(sym_amt, key=lambda x: [get_el_sp(x).X, x])
D:\app\anaconda3\envs\cgdit\lib\site-packages\pymatgen\core\composition.py:373: UserWarning: No Pauling electronegativity for Ne. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  syms = sorted(sym_amt, key=lambda sym: get_el_sp(sym).X)
D:\app\anaconda3\envs\cgdit\lib\site-packages\pymatgen\core\composition.py:1377: UserWarning: No Pauling electronegativity for Ne. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  if len(syms) >= 3 and get_el_sp(syms[-1]).X - get_el_sp(syms[-2]


已按对称性从低到高排序，正在提取 4 个代表性结构至: E:\WORKSPACE\CodePlace\test_CGDiT\output\singlerun\2026-03-10\15-50-41-mp_20\uncondiction_example

[LowestSym] 空间群: 1 (P1) -> LowestSym_sg1_P1_idx0.cif
[LowMidSym] 空间群: 8 (Cm) -> LowMidSym_sg8_Cm_idx7627.cif
[HighMidSym] 空间群: 143 (P3) -> HighMidSym_sg143_P3_idx9070.cif
[HighestSym] 空间群: 229 (Im-3m) -> HighestSym_sg229_Im-3m_idx9991.cif


In [3]:
import os
import random
import torch
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import defaultdict
import traceback
import uuid
import time

# 环境配置：强制使用 Agg 后端
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
matplotlib.use('Agg')

from pymatgen.core import Structure, Lattice
from pymatgen.analysis.structure_matcher import StructureMatcher
from pymatgen.analysis.phase_diagram import PhaseDiagram, PDEntry
from pymatgen.ext.matproj import MPRester
import matgl

class CrystalEvaluator:
    def __init__(self, mp_api_key=None, mlip_model_name="TensorNet-MatPES-r2SCAN-v2025.1-PES"):
        self.mp_api_key = mp_api_key
        self.matcher = StructureMatcher(ltol=0.2, stol=0.3, angle_tol=5.0)

        if any(x in mlip_model_name for x in ["M3GNet", "CHGNet", "MEGNet"]):
            matgl.set_backend("DGL")
        else:
            matgl.set_backend("PYG")

        try:
            self.potential = matgl.load_model(mlip_model_name)
            print(f">>> 成功加载力场模型: {mlip_model_name}")
        except Exception as e:
            print(f">>> 模型加载失败: {e}")
            raise

    def tensors_to_structures(self, pt_file_path):
        if not os.path.exists(pt_file_path):
            return [], 0
        data = torch.load(pt_file_path, map_location='cpu', weights_only=False)
        frac_coords = data['frac_coords']
        atom_types = data['atom_types']
        num_atoms = data['num_atoms'].tolist()
        split_coords = torch.split(frac_coords, num_atoms)
        split_types = torch.split(atom_types, num_atoms)
        structures = []
        for i in range(len(num_atoms)):
            try:
                if 'lattices' in data:
                    lattice = Lattice(data['lattices'][i].cpu().numpy())
                elif 'lengths' in data and 'angles' in data:
                    abc = data['lengths'][i].cpu().numpy()
                    ang = data['angles'][i].cpu().numpy()
                    lattice = Lattice.from_parameters(*abc, *ang)
                else: continue
                struct = Structure(lattice=lattice, species=split_types[i].cpu().numpy() + 1,
                                 coords=np.mod(split_coords[i].cpu().numpy(), 1.0), coords_are_cartesian=False)
                structures.append(struct)
            except: continue
        return structures, len(num_atoms)

    def build_phase_diagram(self, elements_tuple, retries=2):
        if not self.mp_api_key: return None
        for i in range(retries):
            try:
                with MPRester(self.mp_api_key) as mpr:
                    entries = mpr.get_entries_in_chemsys(list(elements_tuple))
                    if not entries: return None
                    return PhaseDiagram(entries)
            except:
                if i < retries - 1:
                    time.sleep(2)
                    continue
                return None

    def evaluate_static_stability(self, structure: Structure, phase_diagram: PhaseDiagram = None):
        try:
            if hasattr(self.potential, "model") and hasattr(self.potential.model, "predict_structure"):
                outputs = self.potential.model.predict_structure(structure)
            elif hasattr(self.potential, "predict_structure"):
                outputs = self.potential.predict_structure(structure)
            else:
                raise AttributeError("Model Error")

            if isinstance(outputs, torch.Tensor):
                total_energy = float(outputs.cpu().numpy())
            elif isinstance(outputs, dict) and 'energies' in outputs:
                total_energy = float(outputs['energies'].cpu().numpy())
            else:
                total_energy = float(outputs[0]) if hasattr(outputs, "__getitem__") else float(outputs)

            e_hull = None
            if phase_diagram:
                entry = PDEntry(structure.composition, total_energy)
                try:
                    e_hull = phase_diagram.get_e_above_hull(entry)
                except:
                    return total_energy, None
            return total_energy, e_hull
        except Exception as e:
            return None, None

    def plot_custom_phase_diagram(self, pd, structure, energy, sys_name, save_dir):
        """完全手动使用 matplotlib 绘制相图，彻底绕过 PDPlotter 的后端限制"""
        try:
            els = sorted([el.symbol for el in pd.elements])
            num_els = len(els)
            unique_id = uuid.uuid4().hex[:4]
            fpath = os.path.join(save_dir, f"PD_{sys_name}_{structure.composition.reduced_formula}_{unique_id}.png")

            plt.figure(figsize=(8, 6))

            # 获取所有稳定项点
            stable_entries = pd.stable_entries

            if num_els == 2:
                # 二元相图：能量 vs 组分
                # 1. 绘制凸包线
                all_pts = []
                for entry in pd.all_entries:
                    x = entry.composition.get_atomic_fraction(els[1])
                    y = pd.get_form_energy_per_atom(entry)
                    all_pts.append((x, y))

                # 绘制所有不稳定相（灰色点）
                pts_arr = np.array(all_pts)
                plt.scatter(pts_arr[:, 0], pts_arr[:, 1], c='lightgray', s=10, alpha=0.5, label='Other phases')

                # 绘制稳定相连线（Convex Hull）
                hull_pts = sorted([(e.composition.get_atomic_fraction(els[1]), pd.get_form_energy_per_atom(e)) for e in stable_entries])
                hx, hy = zip(*hull_pts)
                plt.plot(hx, hy, 'ko-', lw=2, markersize=8, label='Convex Hull')

                # 2. 标注生成的结构
                x_gen = structure.composition.get_atomic_fraction(els[1])
                y_gen = pd.get_form_energy_per_atom(PDEntry(structure.composition, energy))
                e_hull = pd.get_e_above_hull(PDEntry(structure.composition, energy))
                color = 'green' if (e_hull is not None and e_hull <= 0.05) else 'red'

                plt.scatter([x_gen], [y_gen], c=color, marker='*', s=300, edgecolors='black',
                           label=f'Our Structure (+{e_hull:.3f} eV/at)', zorder=10)

                plt.xlabel(f"Fraction of {els[1]}")
                plt.ylabel("Formation Energy (eV/atom)")
                plt.xlim(-0.05, 1.05)

            elif num_els == 3:
                # 三元相图：等边三角形投影
                def get_tri_coords(comp):
                    c = [comp.get_atomic_fraction(e) for e in els]
                    return c[1] + 0.5 * c[2], c[2] * np.sqrt(3) / 2

                # 绘制三角形边界
                plt.plot([0, 1, 0.5, 0], [0, 0, np.sqrt(3)/2, 0], 'k-', lw=1.5)
                plt.text(-0.05, -0.05, els[0], fontsize=12, fontweight='bold')
                plt.text(1.02, -0.05, els[1], fontsize=12, fontweight='bold')
                plt.text(0.48, np.sqrt(3)/2 + 0.02, els[2], fontsize=12, fontweight='bold')

                # 绘制稳定相点
                for entry in stable_entries:
                    tx, ty = get_tri_coords(entry.composition)
                    plt.plot(tx, ty, 'ko', markersize=6)

                # 绘制我们的点
                tx_gen, ty_gen = get_tri_coords(structure.composition)
                e_hull = pd.get_e_above_hull(PDEntry(structure.composition, energy))
                color = 'green' if (e_hull is not None and e_hull <= 0.05) else 'red'
                plt.scatter([tx_gen], [ty_gen], c=color, marker='*', s=300, edgecolors='black',
                           label=f'Our Structure (+{e_hull:.3f} eV/at)', zorder=10)

                plt.axis('off')
                plt.gca().set_aspect('equal')

            plt.title(f"Manual Phase Diagram: {sys_name}\nTarget: {structure.composition.reduced_formula}")
            plt.legend(loc='upper right', fontsize='small')
            plt.savefig(fpath, dpi=300, bbox_inches='tight')
            plt.close()
            return fpath
        except Exception as e:
            traceback.print_exc()
            return f"DRAW_ERROR: {e}"

if __name__ == "__main__":
    PT_FILE_PATH = "../output/singlerun/2026-03-10/15-50-41-mp_20/eval_gen_mp_20.pt"
    MP_API_KEY = "4rY9c4pTnlQcq9WdvqldIuEtOFcBxxGk"

    evaluator = CrystalEvaluator(mp_api_key=MP_API_KEY)
    all_structs, _ = evaluator.tensors_to_structures(PT_FILE_PATH)
    binary_ternary_structs = [s for s in all_structs if 2 <= len(s.composition.elements) <= 3]

    sampled = random.sample(binary_ternary_structs, min(len(binary_ternary_structs), 10))
    pd_dir = os.path.join(os.getcwd(), "Stability_Analysis", "phase_diagrams_static")
    os.makedirs(pd_dir, exist_ok=True)

    print(f"\n>>> 结果将保存至 (手动绘图模式): {pd_dir}")

    for s in tqdm(sampled, desc="处理进度"):
        formula = s.composition.reduced_formula
        els = tuple(sorted([el.symbol for el in s.composition.elements]))
        pd_diag = evaluator.build_phase_diagram(els)
        if not pd_diag: continue

        energy, e_hull = evaluator.evaluate_static_stability(s, pd_diag)
        if energy is not None:
            save_res = evaluator.plot_custom_phase_diagram(pd_diag, s, energy, "-".join(els), pd_dir)
            if "/" in save_res or "\\" in save_res:
                print(f"\n[成功] {formula}: 相图已生成")
            else:
                print(f"\n[失败] {formula}: {save_res}")

    print(f"\n任务全部结束。")

>>> 成功加载力场模型: TensorNet-MatPES-r2SCAN-v2025.1-PES

>>> 结果将保存至 (手动绘图模式): E:\WORKSPACE\CodePlace\test_CGDiT\scripts\Stability_Analysis\phase_diagrams_static


处理进度:  10%|█         | 1/10 [00:02<00:19,  2.12s/it]


[成功] CeZnCu: 相图已生成


处理进度:  20%|██        | 2/10 [00:03<00:13,  1.69s/it]


[成功] TaNb2O4: 相图已生成


...


[成功] DyRu2: 相图已生成


处理进度:  90%|█████████ | 9/10 [00:10<00:00,  1.02it/s]


[成功] ZrGa2Co3: 相图已生成


处理进度: 100%|██████████| 10/10 [00:12<00:00,  1.29s/it]


[成功] Co2O3F: 相图已生成

任务全部结束。
